# XTX / SANOS Vol Surface Case Study

This notebook is the final visual case-study companion for the project. It does not replace `option_surface_step1.ipynb`; instead, it gathers the main diagnostics, remediation results, and multi-day regime evidence into one readable path.

Main paper: `xtx vol surface.pdf`  
Main project notes: `README.md`  
Reflections: `SANOS_XTX_REFLECTIONS.md`

## Reproduce

Run the final case-study pipeline from a terminal:

```bash
python reproduce_xtx_sanos_project.py --trade-date 2025-06-02
```

The heavier multi-day ETA study can be reproduced explicitly:

```bash
python reproduce_xtx_sanos_project.py --multi-day-regime
```

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
DIAG = ROOT / "xtx_diagnostics"
AUTO = DIAG / "auto_regime"
MULTI = DIAG / "multi_day_eta"

def show_markdown(path):
    display(Markdown(path.read_text(encoding="utf-8")))

def show_image(path, width=1000):
    display(Image(filename=str(path), width=width))

## Baseline Joint LP Diagnostics

The baseline surface uses market-only anchors and the original notebook's `ETA=0.25`. The primary fit object is pure call price, while OTM IV is a diagnostic transformation.

In [ ]:
show_markdown(DIAG / "diagnostic_summary.md")

In [ ]:
show_image(DIAG / "05_pure_call_surface.png", width=900)
show_image(DIAG / "06_otm_iv_surface.png", width=900)

## Expiry Case Forensics

The forensic scripts analyze any target DTE via `--target-dte`; this notebook uses 74D as the default project example because it was the main weak point in the 2025-06-02 joint LP. Low coverage does not by itself prove that the market quote is wrong, so we inspect same-expiry shape, nearby calendar consistency, and remediation behavior.

In [ ]:
show_markdown(DIAG / "expiry_case_static_shape_summary.md")
show_markdown(DIAG / "expiry_case_joint_candidate_sweep_summary.md")

In [ ]:
show_image(DIAG / "13_expiry_case_static_shape_diagnostics.png", width=900)
show_image(DIAG / "18_expiry_case_joint_candidate_sweep_metrics.png", width=900)
show_image(DIAG / "19_expiry_case_joint_candidate_fit_curves.png", width=900)

## Automatic Regime-Based Remediation

The final engineering rule is not to densify every expiry. Instead, start with the baseline joint LP, detect weak expiries, and apply dense anchors plus smaller ETA only where the acceptance gates pass.

In [ ]:
show_markdown(AUTO / "auto_regime_summary.md")

In [ ]:
show_image(AUTO / "auto_regime_before_after.png", width=1000)
pd.read_csv(AUTO / "auto_regime_steps.csv")

## Multi-Day ETA Regime Evidence

The multi-day sweep supports the practical rule that anchor density and ETA should be chosen together: denser anchors usually call for smaller ETA, while sparse/longer-dated expiries are the main candidates for remediation.

In [ ]:
show_markdown(MULTI / "multi_day_eta_regime_summary.md")

In [ ]:
show_image(MULTI / "multi_day_eta_regime_charts.png", width=1000)

## Final Practical Rule

```text
Start with market-only joint LP.
Use ETA=0.25 as a benchmark, not as a universal production default.
If an expiry has low coverage or high outside error, diagnose it.
If it is sparse, medium/long dated, or joint-calendar constrained, try dense_full_tail anchors.
For dense remediation, use ETA candidates around 0.01 to 0.10.
Accept remediation only if target fit improves, global coverage is not materially damaged, and calendar checks remain clean.
Optionally run final wing-weighted QP to stabilize q densities.
```